In [51]:


import torch
import torch.nn as nn
from torch.utils.data import Dataset

class BillingualDataset(Dataset):

    def __init__(self, ds, tokenizer_src, tokenizer_tgt, src_lang, tgt_lang, seq_len) -> None:
        super().__init__()

        self.ds = ds
        self.tokenizer_src = tokenizer_src
        self.tokenizer_tgt = tokenizer_tgt
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.seq_len = seq_len

        self.sos_token = torch.tensor([tokenizer_src.token_to_id('<SOS>')], dtype=torch.int64)
        self.eos_token = torch.tensor([tokenizer_src.token_to_id('<EOS>')], dtype=torch.int64)
        self.pad_token = torch.tensor([tokenizer_src.token_to_id('<PAD>')], dtype=torch.int64)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, index):
        src_target_pair = self.ds[index]
        src_text = src_target_pair[self.src_lang]
        tgt_text = src_target_pair[self.tgt_lang]

        enc_input_tokens = self.tokenizer_src.encode(src_text).ids
        dec_input_tokens = self.tokenizer_tgt.encode(tgt_text).ids

        enc_num_padding_tokens = self.seq_len - len(enc_input_tokens) - 2
        dec_num_padding_tokens = self.seq_len - len(dec_input_tokens) - 1

        if enc_num_padding_tokens < 0 or dec_num_padding_tokens < 0:
            raise ValueError('too long')

        encoder_input = torch.cat(
            [
                self.sos_token,
                torch.tensor(enc_input_tokens, dtype=torch.long),
                self.eos_token,
                torch.tensor([self.pad_token] * enc_num_padding_tokens, dtype=torch.long)
            ]
        )

        decoder_input = torch.cat(
            [
                self.sos_token,
                torch.tensor(dec_input_tokens, dtype=torch.long),
                torch.tensor([self.pad_token] * dec_num_padding_tokens, dtype=torch.long)
            ]
        )

        label = torch.cat(
            [
                torch.tensor(dec_input_tokens, dtype=torch.long),
                self.eos_token,
                torch.tensor([self.pad_token] * dec_num_padding_tokens, dtype=torch.long)

            ]
        )

        assert encoder_input.shape[0] == self.seq_len
        assert decoder_input.shape[0] == self.seq_len
        assert label.shape[0] == self.seq_len

        return {
            'encoder_input': encoder_input,
            'decoder_input': decoder_input,
            'encoder_mask': (encoder_input != self.pad_token).unsqueeze(0).unsqueeze(0).int(),
            'decoder_mask': (decoder_input != self.pad_token).unsqueeze(0).unsqueeze(0).int() & compute_casual_mask(decoder_input.shape[0]),
            'label': label,
            'src_text': src_text,
            'tgt_text': tgt_text
        }

def compute_casual_mask(size):
    mask = torch.triu(torch.ones(1, size, size), diagonal=1).type(torch.int)
    return mask == 0


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

from dataset import compute_casual_mask
from vanilla_transformer import build_transformer

from config import get_config, get_weigths_file_path

from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

from torch.utils.tensorboard import SummaryWriter

from tqdm import tqdm
import warnings
from pathlib import Path

def greedy_decode(model, source, source_mask, tokenizer_src, tokenizer_tgt, max_len, device):
    sos_idx = tokenizer_tgt.token_to_id('<SOS>')
    eos_idx = tokenizer_tgt.token_to_id('<EOS>')

    encoder_output = model.encode(source, source_mask)

    decoder_input = torch.empty(1, 1).fill_(sos_idx).type_as(source).to(device)
    while True:
        if decoder_input.shape[1] == max_len:
            break

        decoder_mask = compute_casual_mask(decoder_input.shape[1]).type_as(source_mask).to(device)

        decoder_output = model.decode(encoder_output, source_mask, decoder_input, decoder_mask) # [1, 1, E]

        probs = model.projection(decoder_output[:, -1])

        _, next_word = torch.max(probs, dim=1)
        decoder_input = torch.cat(
            [
                decoder_input,
                torch.empty(1,1).type_as(source).fill_(next_word.item()).to(device)
            ], dim=1
        )

        if next_word == eos_idx:
            break

    return decoder_input.squeeze(0)

def run_validation(model, validation_ds, tokeniozer_src, tokenizer_tgt, max_len, device, print_msg, global_state, writer, num_examples=2):
    model.eval()
    count = 0

    console_width = 50

    with torch.no_grad():
        for batch in validation_ds:
            count += 1
            encoder_input = batch['encoder_input'].to(device)
            encoder_mask = batch['encoder_mask'].to(device)

            assert encoder_input.shape[0] == 1, "Batch_size must be 1"

            model_output = greedy_decode(model, encoder_input, encoder_mask, tokeniozer_src, tokenizer_tgt, max_len, device)
            source_text = batch['src_text'][0]
            target_text = batch['tgt_text'][0]
            model_out_text = tokenizer_tgt.decode(model_output.detach().cpu().numpy())

            print_msg('-'*console_width)
            print_msg(f'SOURCE: {source_text}')
            print_msg(f'TARGET: {target_text}')
            print_msg(f'PREDICTED: {model_out_text}')

            if count == num_examples:
                break


def get_all_sentences(ds, lang):
    for item in ds['train'][lang]:
        yield item

def get_or_build_tokenizer(config, ds, lang):
    tokenizer_path = Path(config['tokenizer_file'].format(lang))
    if not Path.exists(tokenizer_path):
        tokenizer = Tokenizer(WordLevel(unk_token='<UNK>'))
        tokenizer.pre_tokenizer = Whitespace()
        trainer = WordLevelTrainer(special_tokens=['<UNK>', '<PAD>', '<SOS>', '<EOS>'], min_frequency=2)
        tokenizer.train_from_iterator(get_all_sentences(ds, lang), trainer=trainer)
        tokenizer.save(str(tokenizer_path))
    else:
        tokenizer = Tokenizer.from_file(str(tokenizer_path))
    return tokenizer

def get_ds(config):
    ds_raw = load_dataset('bentrevett/multi30k')

    tokenizer_src = get_or_build_tokenizer(config, ds_raw, config['lang_src'])
    tokenizer_tgt = get_or_build_tokenizer(config, ds_raw, config['lang_tgt'])

    train_ds_size = int(0.9 * len(ds_raw['train']))
    val_ds_size = len(ds_raw['train']) - train_ds_size
    print(train_ds_size, )
    train_ds_raw, val_ds_raw = random_split(ds_raw['train'], [train_ds_size, val_ds_size])
    print(list(train_ds_raw))

    train_ds = BillingualDataset(train_ds_raw,
                                 tokenizer_src,
                                 tokenizer_tgt,
                                 config['lang_src'],
                                 config['lang_tgt'],
                                 config['seq_len'])

    val_ds   = BillingualDataset(val_ds_raw,
                                 tokenizer_src,
                                 tokenizer_tgt,
                                 config['lang_src'],
                                 config['lang_tgt'],
                                 config['seq_len'])
    max_len_src = 0
    max_len_tgt = 0
    for item in ds_raw['train']:
        src_ids = tokenizer_src.encode(item[config['lang_src']]).ids
        tgt_ids = tokenizer_tgt.encode(item[config['lang_tgt']]).ids
        max_len_src = max(max_len_src, len(src_ids))
        max_len_tgt = max(max_len_tgt, len(tgt_ids))
    print(f'Max length of source sentence {max_len_src}')
    print(f'Max length of source sentence {max_len_tgt}')

    train_dataloader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
    val_dataloader = DataLoader(val_ds, batch_size=1, shuffle=True)

    return train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt

In [53]:
from config import get_config
config = get_config()
train_ds, train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_ds(config)

26100
[{'en': 'Two men in orange safety vests paint a curb white.', 'de': 'Zwei Männer in orangen Sicherheitswesten malen einen Bordstein weiß an.'}, {'en': 'A crowd of people sit at round tables in a banquet hall.', 'de': 'Viele Personen sitzen in einem Festsaal an runden Tischen.'}, {'en': 'Two girls eating a meal together.', 'de': 'Zwei Mädchen essen zusammen eine Mahlzeit.'}, {'en': 'A man is having a walk with his dog during sunset.', 'de': 'Ein Mann geht mit seinem Hund bei Sonnenuntergang Gassi.'}, {'en': 'Three people sitting on a bus each doing something different.', 'de': 'Drei Personen sitzen in einem Bus und tun alle unterschiedliche Dinge.'}, {'en': 'A man with an unusual hairstyle in a t-shirt carries two plates of food.', 'de': 'Ein Mann der eine ungewöhnlichen Frisur und ein T-Shirt trägt, trägt zwei Teller mit Essen.'}, {'en': 'A little girl and boy dressed in red eat cereal out of purple bowls.', 'de': 'Ein kleines Mädchen und ein kleiner Junge in roter Kleidung essen

In [54]:
train_ds[0]

{'encoder_input': tensor([   2,   21,   32,    7,  459, 2418, 1545,   20, 1848,  190,   23,    4,
            3,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
            1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,    1,
           

In [22]:
for i in train_dataloader:
    print(i)

KeyError: "Invalid key: 1. Please first select a split. For example: `my_dataset_dictionary['train'][1]`. Available splits: ['test', 'train', 'validation']"